# 17 — UR5e trajectory optimization with obstacle avoidance

**Objectives.** Form a six-axis joint-space OCP, constrain endpoints and limits, add a soft Cartesian obstacle, and warm-start mesh refinements. Dynamics are $\dot q=v$, $\dot v=a$; acceleration is the control. This surrogate plans smooth joint trajectories but does not predict motor torque.


In [ ]:
import casadi as ca
import numpy as np
from casadi_tutorial.models.ur5e import UR5eParameters,forward_kinematics
p=UR5eParameters(); N=15; duration=3.0; dt=duration/N
q0=np.zeros(6); q_goal=np.array([0,-np.pi/3,np.pi/2,-np.pi/3,np.pi/2,0])
q_symbol=ca.MX.sym('q',6); tool=ca.Function('tool',[q_symbol],[forward_kinematics(q_symbol,p)[:3,3]])
center=0.5*(np.asarray(tool(q0).full()).ravel()+np.asarray(tool(q_goal).full()).ravel()); radius=0.08
opti=ca.Opti(); Q=opti.variable(6,N+1); V=opti.variable(6,N+1); A=opti.variable(6,N); slack=opti.variable(1,N+1)
opti.subject_to(Q[:,0]==q0); opti.subject_to(Q[:,-1]==q_goal); opti.subject_to(V[:,0]==0); opti.subject_to(V[:,-1]==0)
for k in range(N):
    opti.subject_to(Q[:,k+1]==Q[:,k]+dt*V[:,k]+0.5*dt**2*A[:,k])
    opti.subject_to(V[:,k+1]==V[:,k]+dt*A[:,k])
opti.subject_to(opti.bounded(p.q_min[:,None],Q,p.q_max[:,None])); opti.subject_to(opti.bounded(-p.v_max[:,None],V,p.v_max[:,None])); opti.subject_to(opti.bounded(-p.a_max[:,None],A,p.a_max[:,None])); opti.subject_to(slack>=0)
for k in range(N+1): opti.subject_to(ca.sumsqr(tool(Q[:,k])-center)+slack[0,k]>=radius**2)
opti.minimize(dt*ca.sumsqr(A)+1e4*ca.sumsqr(slack)+1e-2*ca.sumsqr(V))
for joint in range(6): opti.set_initial(Q[joint,:],np.linspace(q0[joint],q_goal[joint],N+1))
opti.solver('ipopt',{'print_time':False},{'print_level':0,'max_iter':500})
sol=opti.solve(); print('objective',sol.value(opti.f),'max slack',np.max(sol.value(slack)))
assert sol.stats()['success'] and np.max(sol.value(slack)) < radius**2


In [ ]:
decision=ca.vertcat(ca.vec(Q),ca.vec(V),ca.vec(A),ca.vec(slack))
J=ca.jacobian(opti.g,decision)
print('constraint Jacobian shape',J.shape,'nnz',J.nnz(),'density',J.nnz()/(J.size1()*J.size2()))


A soft obstacle preserves feasibility while exposing collision pressure through slack. Production planning must check full link geometry continuously between nodes and use certified collision models. For mesh refinement, interpolate $Q,V,A$ onto the finer grid, seed the new problem, and compare objective and between-node clearance.

**Common mistakes:** checking only the tool point, trusting nodes between coarse intervals, and claiming dynamic feasibility from the surrogate. **Recap:** encode honest model limits, examine sparsity, and validate the continuous trajectory outside the NLP. [Exercises](../exercises/06_ur5e.ipynb).
